# Link para utilizar `amostra_dataset_reduzida`:
https://drive.google.com/file/d/1ece3JbpI37I8OzFkB5eF8b3kDrYWFF-1/view?usp=sharing

### Integrantes:
- Denis Hitarley Santos Farias
- Emily da Cunha Vasconcelos
- Juan Pablo Cabrera Vidal
- Karina Hae Eun Huh

In [ ]:
%pip install pandas matplotlib seaborn

### Exercício 1
#### Considere o dataset selecionado. Faça a caracterização inicial dos principais tipos dos atributos e avalie as Medidas de Localidade.

In [7]:
import pandas as pd
from IPython.display import display

df = pd.read_csv("amostra_dataset_reduzida.csv")

cpu_cols = [col for col in df.columns if "cpu_usage" in col]
mem_cols = [col for col in df.columns if "memory_usage" in col]
lat_cols = [col for col in df.columns if col.endswith("_latency")]

df_agg = pd.DataFrame()
df_agg["cpu"] = df[cpu_cols].mean(axis=1)
df_agg["memory"] = df[mem_cols].mean(axis=1) / (
    1024 * 1024
)  # convertido de Bytes para MB
df_agg["latencia_ms"] = df[lat_cols].mean(axis=1) / 1000

tipos_de_dados = pd.DataFrame(
    {
        "Colunas": ["cpu", "memory", "latencia_ms"],
        "Tipo Estatístico": [
            "Quantitativo Contínuo",
            "Quantitativo Contínuo",
            "Quantitativo Contínuo",
        ],
    }
)

medidas_localidade = pd.DataFrame(
    {"Média": df_agg.mean(), "Mediana": df_agg.median(), "Moda": df_agg.mode().iloc[0]}
)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Caracterização dos Dados:")
display(tipos_de_dados)

print("Medidas de Localidade:")
display(medidas_localidade)

Caracterização dos Dados:


,Colunas,Tipo Estatístico
0,cpu,Quantitativo Contínuo
1,memory,Quantitativo Contínuo
2,latencia_ms,Quantitativo Contínuo


Medidas de Localidade:


,Média,Mediana,Moda
cpu,270.22,187.86,155.99
memory,24.28,0.00,0.00
latencia_ms,37.29,12.63,6.58


- **Consumo de CPU (cpu):** O consumo médio de CPU por serviço apresentou variação, enquanto metade das execuções registrou até 187,86 segundos de uso de CPU (mediana), a média foi puxada para 270,22 segundos. Isso indica que alguns cenários de estresse aumentaram o consumo de processamento. O valor mais frequente (moda) foi de 155,99 segundos.

- **Consumo de Memória (memory):** Na maior parte das execuções, não foi observado consumo extra relevante de memória nos serviços, o que é evidenciado pela mediana e moda permanecerem em 0,00 MB. A média atingiu aproximadamente 24,28 MB, refletindo a influência dos cenários com injeção de falhas de memória que elevaram o consumo pontualmente.

- **Latência Média (latencia_ms):** A latência média por trace, utilizada como proxy de volumetria de requisições, apresentou mediana de 12,63 ms e média de 37,29 ms. A diferença entre média e mediana evidencia a presença de valores discrepantes (outliers): durante os cenários de bottleneck ativo, a latência pode ultrapassar 8.000 ms, puxando a média para cima.

---------------------

### Exercício 2
### Analise o espalhamento das observações dos atributos de pelo menos 2 atributos. Utilize técnicas gráficas como Boxplots para verificar a presença de outliers. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

df = pd.read_csv("amostra_dataset_reduzida.csv")

cpu_cols = [col for col in df.columns if "cpu_usage" in col]
mem_cols = [col for col in df.columns if "memory_usage" in col]
lat_cols = [col for col in df.columns if col.endswith("_latency")]

df_agg = pd.DataFrame()
df_agg["cpu"] = df[cpu_cols].mean(axis=1)
df_agg["memory"] = df[mem_cols].mean(axis=1) / (
    1024 * 1024
)  # convertido de Bytes para MB
df_agg["latencia_ms"] = df[lat_cols].mean(axis=1) / 1000

# Medidas de Espalhamento
intervalo = df_agg.max() - df_agg.min()
variancia = df_agg.var()
desvio_padrao = df_agg.std()

# Calculando o IQR
q1 = df_agg.quantile(0.25)
q3 = df_agg.quantile(0.75)
iqr = q3 - q1

medidas_espalhamento = pd.DataFrame(
    {
        "Intervalo (Max-Min)": intervalo,
        "Variância": variancia,
        "Desvio Padrão": desvio_padrao,
        "IQR (Q3 - Q1)": iqr,
    }
)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Medidas de Espalhamento: ")
display(medidas_espalhamento)

# Visualização com Boxplots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.boxplot(y=df_agg["cpu"], ax=axes[0], color="skyblue")
axes[0].set_title("CPU (s)")
axes[0].set_ylabel("Segundos")

sns.boxplot(y=df_agg["memory"], ax=axes[1], color="lightgreen")
axes[1].set_title("Memória (MB)")
axes[1].set_ylabel("MB")

sns.boxplot(
    y=df_agg["latencia_ms"], ax=axes[2], color="salmon"
)  # corrigido: latencia_ms
axes[2].set_title("Latência Média (ms)")
axes[2].set_ylabel("Milissegundos (ms)")

plt.tight_layout()
plt.show()

- **Boxplot de CPU:** O gráfico mostra que a maior parte das execuções apresentou um consumo médio de CPU concentrado abaixo de 350 segundos, porém alguns testes registraram valores muito superiores, representados pelos pontos que chegam a picos de até 933,44 segundos. Esses casos representam cenários de estresse severos, nos quais as falhas injetadas aumentaram significativamente a demanda por processamento em comparação ao comportamento normal do sistema.

- **Boxplot de Memória:** O consumo extra de memória permaneceu em 0 MB na maior parte das execuções (IQR de 0 a 12,45 MB), indicando que o sistema normalmente não sofre impacto nesse recurso. Porém a enorme quantidade de pontos no gráfico evidencia que testes específicos geraram picos que alcançaram até 176,01 MB. Esses eventos correspondem aos cenários em que foram simuladas falhas diretas de memória, produzindo aumentos pontuais e significativamente superiores ao padrão.

- **Boxplot de Latência:** A latência média por trace, utilizada como proxy de volumetria de requisições, permaneceu baixa na maior parte dos cenários (mediana de 12,63 ms, IQR de 2,84 ms a 20,02 ms). A linha superior estendida e os pontos acima dos bigodes evidenciam picos que chegam a 8.681,81 ms durante os cenários de bottleneck ativo, demonstrando que a sobrecarga de requisições impacta severamente o tempo de resposta dos microsserviços.

-------------

### Exercício 3
#### Faça a análise de correlação e covariância entre os atributos.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv("amostra_dataset_reduzida.csv")

cpu_cols = [col for col in df.columns if "cpu_usage" in col]
mem_cols = [col for col in df.columns if "memory_usage" in col]
lat_cols = [col for col in df.columns if col.endswith("_latency")]

df_agg = pd.DataFrame()
df_agg["cpu"] = df[cpu_cols].mean(axis=1)
df_agg["memory_mb"] = df[mem_cols].mean(axis=1) / (1024 * 1024)  # Bytes para MB
df_agg["latencia_ms"] = df[lat_cols].mean(axis=1) / 1000

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Matriz de Covariância: ")
display(df_agg.cov())

corr_pearson = df_agg.corr(method="pearson")
corr_spearman = df_agg.corr(method="spearman")

print("Matriz de Correlação de Pearson:")
display(corr_pearson)

print("Matriz de Correlação de Spearman:")
display(corr_spearman)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mapa 1: Pearson
sns.heatmap(
    corr_pearson, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, ax=axes[0]
)
axes[0].set_title("Correlação Pearson")

# Mapa 2: Spearman
sns.heatmap(
    corr_spearman, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, ax=axes[1]
)
axes[1].set_title("Correlação Spearman")

plt.tight_layout()
plt.show()

##### **1. Análise da Matriz de Covariância**
Ao analisar os resultados, observamos que os valores são positivos entre CPU e memória, o que indica que tendem a variar na mesma direção. A covariância entre CPU e latência é negativa (-1.766,65), sugerindo que em alguns cenários o aumento de CPU está associado à redução de latência — possivelmente porque mais CPU disponível acelera o processamento. Contudo, a covariância sofre influência da escala e das unidades de medida, portanto a análise da correlação é a métrica mais adequada para entender o grau de relacionamento entre as variáveis.

##### **2. Análise das Matrizes de Correlação**
- **Correlação CPU vs. Memória:** A correlação de Pearson indicou uma relação forte entre CPU e memória (0,86), mas a correlação de Spearman mostrou uma relação muito fraca (0,10). Isso sugere que alguns casos extremos de teste tiveram grande influência nos resultados. O aumento do uso de CPU não está necessariamente acompanhado por um aumento semelhante no consumo de memória durante o funcionamento normal do sistema.

- **Correlação Memória vs. Latência:** A correlação de Spearman entre memória e latência foi de -0,44 (moderada negativa), indicando que quando a memória está mais pressionada, a latência tende a ser menor — o que pode refletir que os experimentos de bottleneck de memória foram conduzidos com cargas de requisições distintas dos experimentos de CPU. A correlação de Pearson (-0,07) é próxima de zero, confirmando que a relação não é linear.

- **Correlação CPU vs. Latência:** Tanto Pearson (-0,05) quanto Spearman (0,07) indicam correlação próxima de zero, sugerindo que o consumo acumulado de CPU, por ser um contador monotônico desde a inicialização do container, não representa diretamente a pressão instantânea de carga sobre o serviço.